# Microsoft Foundry — Models (`project_client.beta.models`)

This notebook demonstrates how to use the synchronous `.beta.models` operations on `AIProjectClient` to register a local model with the project, list and inspect model versions, retrieve storage credentials, update metadata, and delete a model version.

## Operations covered

| Method | Purpose |
| --- | --- |
| `register_model` | End-to-end helper: provisions blob storage, uploads weights via `azcopy`, and commits the registration. **Recommended entry point.** |
| `list` | List the latest version of every registered model. |
| `list_versions` | List all versions of a single model. |
| `get` | Fetch a specific `ModelVersion`. |
| `get_credentials` | Retrieve a SAS URI for the blob container backing a registered version. |
| `update` | Merge-patch the `description` / `tags` of an existing model version. |
| `delete` | Delete a specific model version. |
| `pending_upload` / `create_async` | Lower-level building blocks used internally by `register_model`. |

## Prerequisites

```bash
pip install "azure-ai-projects>=2.2.0" azure-identity python-dotenv
```

[**AzCopy**](https://aka.ms/downloadazcopy) must be installed and on `PATH` (used by `register_model` to upload weight files):

```powershell
winget install --id Microsoft.Azure.AZCopy.10 -e
```

Set the following environment variables (a `.env` file is supported):

* `FOUNDRY_PROJECT_ENDPOINT` — required. The Azure AI Project endpoint shown on the Microsoft Foundry project Overview page.
* `MODEL_NAME` — optional. Defaults to `"sample-model"`.
* `MODEL_VERSION` — optional. Defaults to `"1"`.

## 1. Set up the client

Create an `AIProjectClient` using `DefaultAzureCredential`. We also create a tiny temp folder with two dummy "weight" files to register.

In [ ]:
import os
import pathlib
import tempfile

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model_name = os.environ.get("MODEL_NAME", "sample-model")
model_version = os.environ.get("MODEL_VERSION", "1")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

# Build a tiny throw-away source folder with two files to upload.
source_dir = pathlib.Path(tempfile.mkdtemp(prefix="sample-model-"))
(source_dir / "weights.bin").write_bytes(b"hello-foundry-model")
(source_dir / "config.json").write_text('{"sample": true}')
print(f"Source folder: {source_dir}")
for f in sorted(source_dir.rglob("*")):
    if f.is_file():
        print(f"  - {f.relative_to(source_dir)} ({f.stat().st_size} bytes)")

## 2. Register a local model — `register_model`

`register_model` is the recommended way to create a new model version from local files. It packs the spec's three required steps into a single call:

1. `pending_upload(...)` &mdash; provision a project-managed blob container and obtain a SAS URI.
2. `azcopy copy ...` &mdash; upload the local weight files directly to that container.
3. `create_async(...)` &mdash; commit the registration, then poll `get()` until the new `ModelVersion` is observable.

The `weight_type` parameter accepts the `FoundryModelWeightType` enum (`FULL_WEIGHT`, `LO_RA`, `DRAFT_MODEL`) or the equivalent string. The committed `ModelVersion` also exposes service-populated fields such as `artifact_profile`, `source`, and `warnings`.

Use the lower-level `pending_upload` / `create_async` methods directly only if you need to customize one of those steps (for example, to use your own upload tool).

In [ ]:
from azure.ai.projects.models import FoundryModelWeightType

model = project_client.beta.models.register_model(
    name=model_name,
    version=model_version,
    source=source_dir,
    weight_type=FoundryModelWeightType.FULL_WEIGHT,
    description="Sample model registered from sample_models.ipynb",
    tags={"source": "sample_models.ipynb"},
)

print(f"Registered: {model.name}@{model.version}")
print(f"  id               = {model.id}")
print(f"  blob_uri         = {model.blob_uri}")
print(f"  weight_type      = {model.weight_type}")
print(f"  description      = {model.description}")
print(f"  tags             = {model.tags}")
print(f"  source           = {model.source}")
print(f"  artifact_profile = {model.artifact_profile}")
print(f"  warnings         = {model.warnings}")

## 3. Get a specific model version — `get`

In [ ]:
fetched = project_client.beta.models.get(name=model_name, version=model_version)
print(f"{fetched.name}@{fetched.version} -> {fetched.id}")

## 4. List all versions of a model — `list_versions`

In [ ]:
print(f"Versions of {model_name!r}:")
for mv in project_client.beta.models.list_versions(name=model_name):
    print(f"  - {mv.version}")

## 5. List the latest version of every model — `list`

In [ ]:
print("Latest version of every registered model in this project:")
for mv in project_client.beta.models.list():
    print(f"  - {mv.name}@{mv.version}")

## 6. Get blob credentials for a model version — `get_credentials`

Returns a short-lived SAS URI that can be used to read the blob container backing the registered model. The SAS query string is sensitive &mdash; treat it like a credential.

In [ ]:
from azure.ai.projects.models import ModelCredentialRequest

creds = project_client.beta.models.get_credentials(
    name=model_name,
    version=model_version,
    body=ModelCredentialRequest(blob_uri=model.blob_uri),
)
blob_ref = getattr(creds, "blob_reference_for_consumption", None) or getattr(creds, "blob_reference", None)
if blob_ref is not None:
    sas_uri = (blob_ref.credential.sas_uri if blob_ref.credential else None) or ""
    print(f"  blob_uri = {blob_ref.blob_uri}")
    print(f"  sas_uri  = {sas_uri.split('?', 1)[0]}?<sas-redacted>")
else:
    print(creds)

## 7. Update an existing model version — `update`

`update` issues a merge-patch (`PATCH /models/{name}/versions/{version}`) and accepts an `UpdateModelVersionRequest` body containing only the patchable fields (`description`, `tags`).

In [ ]:
from azure.ai.projects.models import UpdateModelVersionRequest

updated = project_client.beta.models.update(
    name=model_name,
    version=model_version,
    body=UpdateModelVersionRequest(
        description="Updated description",
        tags={"source": "sample_models.ipynb", "updated": "true"},
    ),
)
print(f"Updated: {updated.name}@{updated.version}")
print(f"  description = {updated.description}")
print(f"  tags        = {updated.tags}")

## 8. Delete a model version — `delete`

In [ ]:
project_client.beta.models.delete(name=model_name, version=model_version)
print(f"Deleted {model_name}@{model_version}.")

## Cleanup

Close the client and credential.

In [ ]:
project_client.close()
credential.close()